# 05 — Diagnose and retry the 513 still-missing CPGs

Read only the `still_missing` rows from notebook 04. Write one classification record per ID, then retry with additional attribute definitions and a C++ fallback. Macro-template rows with `##` get only a **synthetic preview**, never an accepted CPG; broken preprocessor fragments are recorded as needing original source context. All outputs are separate and resumable. This notebook does not modify six original CPG JSONL files or existing prompt bundles.


In [1]:
import sys,os,re,json,hashlib,tempfile
from pathlib import Path
from collections import Counter
ROOT=Path('/mnt/d/JUNIOR/LabThayTho/GRACE-summary/01_Author_Baseline_Unified')
sys.path.insert(0,str(ROOT))
from prepare_joern_cpg import extract_batch,run_joern_parse,run_joern_export,read_graphml,source_function_graph
SOURCE=ROOT/'output/local_full/joern_second_recovery_20261031'
OUT=ROOT/'output/local_full/joern_third_recovery_20261031';OUT.mkdir(parents=True,exist_ok=True)
JOERN=Path('/home/yungn/tools/joern/y/joern-cli')
OPTS=os.environ.get('JOERN_JAVA_OPTS','-Xms128m -Xmx1536m')
BATCH_SIZE=10
assert (JOERN/'joern-parse').is_file()
def digest(x):return hashlib.sha256(x.encode()).hexdigest()
def load(path):
    if not path.exists():return []
    with path.open(encoding='utf-8') as f:return [json.loads(x) for x in f if x.strip()]
rows=[]
for d in ('devign','reveal'):
    for s in ('train','validation','test'):
        p=SOURCE/f'{d}_{s}.results.jsonl'
        assert p.exists(),p
        rows += [r for r in load(p) if r['status']=='still_missing']
assert len(rows)==513 and len({r['id'] for r in rows})==513
print('Loaded',len(rows),'distinct IDs')


Loaded 513 distinct IDs


## Per-sample cause classification

The categories are evidence from source text, not a proof of unique root cause. A sample may have more than one problem. Initial mutually exclusive classification (priority: token paste, preprocessor, C++, attribute/type macro, other): Devign **0 / 9 / 0 / 11 / 4**; REVEAL **184 / 38 / 19 / 156 / 92**. `##` indicates an unexpanded C macro template; replacing it with text would invent a function name. Broken `#if/#else/#endif` regions need their original source file/build configuration. C++ signatures may require `.cpp`. Remaining cases are retried and keep a concrete Joern error. `classification.jsonl` contains all 513 IDs.


In [2]:
def classify(src):
    evidence=[]
    for name,pat in [
        ('token_paste',r'##'),('preprocessor_fragment',r'(?m)^\s*#\s*(?:if|else|endif|define|include)\b|#\s+(?:if|else|endif)\b'),
        ('cpp_signature',r'::|\b(?:ostream|istream|template|class)\b|\b\w+\s*&\s*\w+'),
        ('missing_attribute_or_type_macro',r'\b(?:GCC_FMT_ATTR|QEMU_WARN_UNUSED_RESULT|av_restrict|av_noreturn|HB_UNUSED|ATTRIBUTE_UNUSED|__user|__always_inline|CONCAT|hidden_def)\b')]:
        if re.search(pat,src):evidence.append(name)
    primary=next((x for x in ('token_paste','preprocessor_fragment','cpp_signature','missing_attribute_or_type_macro') if x in evidence),'unclassified_parse_failure')
    return primary,evidence
classification=OUT/'classification.jsonl'
with classification.open('w',encoding='utf-8') as f:
    for row in rows:
        primary,evidence=classify(row['original_func'])
        f.write(json.dumps({'id':row['id'],'dataset':row['dataset'],'split':row['split'],
            'primary_cause':primary,'evidence':evidence,'func_sha256':digest(row['original_func']),
            'prior_error':row['error']})+'\n')
print('Classification:',classification)
print(Counter((r['dataset'],classify(r['original_func'])[0]) for r in rows))


Classification: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/01_Author_Baseline_Unified/output/local_full/joern_third_recovery_20261031/classification.jsonl
Counter({('reveal', 'token_paste'): 184, ('reveal', 'missing_attribute_or_type_macro'): 145, ('reveal', 'unclassified_parse_failure'): 77, ('reveal', 'cpp_signature'): 45, ('reveal', 'preprocessor_fragment'): 38, ('devign', 'missing_attribute_or_type_macro'): 10, ('devign', 'preprocessor_fragment'): 9, ('devign', 'unclassified_parse_failure'): 4, ('devign', 'cpp_signature'): 1})


## Third retry

The additional prelude defines common source annotations only in temporary parse inputs. `GCC_FMT_ATTR(a,b)` is function-like. If normal C parsing fails and the signature clearly contains C++, the sample is retried as `.cpp`. All recovered graphs are marked `candidate_review`; check method name and body coverage before merging. Rows with `##` are marked `needs_macro_expansion` and **cannot be merged** without the real macro expansion from the source project. Preprocessor fragments are marked `needs_source_context`.


In [3]:
BASE=('av_cold','coroutine_fn','always_inline','av_always_inline','av_noinline','QEMU_NORETURN',
      'attribute_align_arg','av_unused','CUDAAPI','WINAPI','G_GNUC_UNUSED','av_flatten',
      'OPPROTO','STATUS_PARAM','STATUS_VAR','_U_','U_CALLCONV','PNGAPI','TSRMLS_DC','TSRMLS_CC')
EXTRA=('QEMU_WARN_UNUSED_RESULT','av_restrict','av_noreturn','__user','HB_UNUSED',
       'ATTRIBUTE_UNUSED','ossl_inline','__always_inline')
PRELUDE=''.join(f'#ifndef {x}\n#define {x}\n#endif\n' for x in BASE+EXTRA)
PRELUDE+='#ifndef GCC_FMT_ATTR\n#define GCC_FMT_ATTR(a,b)\n#endif\n'
def cpp_like(src):
    head=src.split('{',1)[0]
    return bool(re.search(r'::|\b(?:ostream|istream|template|class)\b|\b\w+\s*&\s*\w+',head))
def parse_cpp(src):
    with tempfile.TemporaryDirectory(prefix='grace-third-cpp-') as t:
        root=Path(t);d=root/'src';d.mkdir();(d/'sample.cpp').write_text(PRELUDE+src,encoding='utf-8')
        env=os.environ.copy();env['JAVA_OPTS']=OPTS
        run_joern_parse(d,root/'cpg.bin',JOERN,env,OPTS)
        run_joern_export(root/'cpg.bin',root/'export',JOERN,env)
        files=list((root/'export').glob('*.xml'))+list((root/'export').glob('*.graphml'))
        if len(files)!=1:raise ValueError(f'Expected one GraphML, got {len(files)}')
        return source_function_graph(*read_graphml(files[0]))
def retry(dataset,split):
    subset=[r for r in rows if r['dataset']==dataset and r['split']==split]
    journal=OUT/f'{dataset}_{split}.results.jsonl'
    old={r['id']:r for r in load(journal)}
    for r in subset:
        if r['id'] in old and (old[r['id']]['func_sha256']!=digest(r['original_func']) or old[r['id']]['prelude_sha256']!=digest(PRELUDE)):
            raise ValueError('Journal input/prelude changed: '+r['id'])
    pending=[r for r in subset if r['id'] not in old]
    print(dataset,split,'total',len(subset),'done',len(old),'pending',len(pending),flush=True)
    with journal.open('a',encoding='utf-8',newline='\n') as out:
        for offset in range(0,len(pending),BATCH_SIZE):
            group=pending[offset:offset+BATCH_SIZE]
            safe=[r for r in group if classify(r['original_func'])[0] not in ('token_paste','preprocessor_fragment')]
            batch=[(i,{'id':r['id'],'func':PRELUDE+r['original_func'],'target':0}) for i,r in enumerate(safe)]
            try: parsed=extract_batch(batch,JOERN,OPTS) if batch else []
            except Exception as exc: parsed=[(i,b,{'nodes':[],'edges':[]},f'{type(exc).__name__}: {str(exc)[:500]}') for i,b in batch]
            byid={b['id']:(g.get('nodes') or [],g.get('edges') or [],e) for _,b,g,e in parsed}
            for r in group:
                src=r['original_func']; cause,evidence=classify(src)
                nodes,edges,error=byid.get(r['id'],([],[],r.get('error')))
                status='candidate_review' if nodes else 'still_missing';mode='c'
                if cause=='preprocessor_fragment':status='needs_source_context';mode='not_run'
                elif cause=='token_paste':
                    status='needs_macro_expansion';mode='not_run'
                elif not nodes and cpp_like(src):
                    try:nodes,edges=parse_cpp(src);status='candidate_review';mode='cpp';error=None
                    except Exception as exc:error=(error or '')+f'; cpp: {type(exc).__name__}: {str(exc)[:250]}'
                result={'id':r['id'],'dataset':dataset,'split':split,'status':status,'cause':cause,
                    'evidence':evidence,'mode':mode,'error':error,'original_func':src,
                    'func_sha256':digest(src),'prelude_sha256':digest(PRELUDE),
                    'nodes':nodes,'edges':edges,'method_names':[n.get('name') for n in nodes if n.get('type')=='METHOD']}
                out.write(json.dumps(result,ensure_ascii=False)+'\n')
            out.flush();os.fsync(out.fileno())
            print(dataset,split,min(offset+BATCH_SIZE,len(pending)),'/',len(pending),flush=True)
    return journal
journals=[retry(d,s) for d in ('devign','reveal') for s in ('train','validation','test')]


devign train total 17 done 0 pending 17
devign train 10 / 17
devign train 17 / 17
devign validation total 4 done 0 pending 4
devign validation 4 / 4
devign test total 3 done 0 pending 3
devign test 3 / 3
reveal train total 397 done 0 pending 397
reveal train 10 / 397
reveal train 20 / 397
reveal train 30 / 397
reveal train 40 / 397
reveal train 50 / 397
reveal train 60 / 397
reveal train 70 / 397
reveal train 80 / 397
reveal train 90 / 397
reveal train 100 / 397
reveal train 110 / 397
reveal train 120 / 397
reveal train 130 / 397
reveal train 140 / 397
reveal train 150 / 397
reveal train 160 / 397
reveal train 170 / 397
reveal train 180 / 397
reveal train 190 / 397
reveal train 200 / 397
reveal train 210 / 397
reveal train 220 / 397
reveal train 230 / 397
reveal train 240 / 397
reveal train 250 / 397
reveal train 260 / 397
reveal train 270 / 397
reveal train 280 / 397
reveal train 290 / 397
reveal train 300 / 397
reveal train 310 / 397
reveal train 320 / 397
reveal train 330 / 397
reve

In [4]:
for p in journals:
    print(p.name,dict(Counter(r['status'] for r in load(p))))
print('Only candidate_review may be considered for manual graph-quality review. needs_macro_expansion and needs_source_context need original project source/build settings.')


devign_train.results.jsonl {'still_missing': 3, 'needs_source_context': 6, 'candidate_review': 8}
devign_validation.results.jsonl {'needs_source_context': 2, 'candidate_review': 2}
devign_test.results.jsonl {'still_missing': 1, 'needs_source_context': 1, 'candidate_review': 1}
reveal_train.results.jsonl {'candidate_review': 125, 'needs_macro_expansion': 146, 'still_missing': 98, 'needs_source_context': 28}
reveal_validation.results.jsonl {'needs_macro_expansion': 21, 'needs_source_context': 7, 'candidate_review': 14, 'still_missing': 6}
reveal_test.results.jsonl {'candidate_review': 11, 'needs_source_context': 3, 'needs_macro_expansion': 17, 'still_missing': 13}
Only candidate_review may be considered for manual graph-quality review. needs_macro_expansion and needs_source_context need original project source/build settings.
